### Problema no lineal
$$\begin{array}{cl}-\Delta u + G(u) = 0 & \text{en } \Omega \\
u = 1 & \text{en }\partial \Omega\end{array}
$$
con $G(u) = e^{\alpha u}$

#### Formulación Variacional no lineal:
$$\int_\Omega \nabla u \nabla v + \int_\Omega G(u)v = 0$$
que escribimos como
$$F(u) = 0$$
con 
$$F(u) = \int_\Omega \nabla u \nabla v + \int_\Omega G(u) v$$
Usaremos el método de Newton para resolver, por lo que será preciso calcular
$$\langle \delta F(u), \delta u\rangle = \int_\Omega \nabla \delta u\, \nabla v + \int_\Omega G'(u) \delta u\, v$$

In [ ]:
%reset  -f
import mfem.ser as mfem
from glvis import glvis # visualización
import numpy as np

In [ ]:
meshfile = 'mallas/star.mesh'
mesh = mfem.Mesh(meshfile)
for i in range(2):
    mesh.UniformRefinement()

dim = mesh.Dimension()

In [ ]:
order = 1
fec = mfem.H1_FECollection(order, dim)

fespace = mfem.FiniteElementSpace(mesh, fec)
print("Número de incógnitas:",fespace.GetTrueVSize())

#### Función no lineal y derivada

In [ ]:
alpha = 0.15
def pFunc(x):
    return np.exp(alpha*x)
#    return x*x

def dpFunc(x):
    return alpha*np.exp(alpha*x)
#    return 2*x

#### Coeficientes para la forma no lineal
Hemos de definir unos coeficientes ligados a la `GridFunction` por lo que debemos usar el método `Eval`

In [ ]:
# Coeficiente G(u)
class NonlinearCoefficient(mfem.PyCoefficientBase):
    def __init__(self, gf):
        super(NonlinearCoefficient, self).__init__(0) # inicialización en la clase padre
        self.gf = gf
    def Eval(self, T, ip):
        val = self.gf.GetValue(T,ip)
        return pFunc(val)

# Coeficiente G'(u)        
class NonlinearDerivativeCoefficient(mfem.PyCoefficientBase):
    def __init__(self, gf):
        super(NonlinearDerivativeCoefficient, self).__init__(0) # inicialización en la clase padre
        self.gf = gf
    def Eval(self, T, ip):
        val = self.gf.GetValue(T,ip)
        return dpFunc(val)

#### Parte no lineal de la formulación variacional
Creamos un `PyNonlinearFormIntegrator` que corresponderá al cálculo de la integral de la parte no lineal. Entonces, $\int_\Omega G(u)v$ se puede ver como un integrador del tipo `DomainLFIntegrator` con coeficiente $G(u)$, mientras que la derivada dará lugar a $\int_\Omega G'(u)\delta u\, v$ que correspondería a un `MassIntegrator` con coeficiente G'(u).

El método `AssembleElementVector`  se encarga de evaluar la parte no lineal de $F(u)$, mientras que `AsssembleElementGrad` evalúa la parte de la derivada

In [ ]:
class NonlinearMassIntegrator(mfem.PyNonlinearFormIntegrator):
    def __init__(self, fes):
        super().__init__()
        self.fes = fes
        self.gf = mfem.GridFunction(fes)

    def AssembleElementVector(self, el, T, elfun, elvect):
        dofs = self.fes.GetElementDofs(T.ElementNo)        
        ardofs = mfem.intArray(dofs)
        # asociamos los valores del vector elfun a la gridfunction gf
        self.gf.SetSubVector(ardofs, elfun)
        # definimos el coeficiente G(gf) para la parte no lineal
        coeff = NonlinearCoefficient(self.gf)
        integ = mfem.DomainLFIntegrator(coeff)
        integ.AssembleRHSElementVect(el,T,elvect)

    def AssembleElementGrad(self, el, T, elfun, elmat):
        dofs = self.fes.GetElementDofs(T.ElementNo)
        ardofs = mfem.intArray(dofs)
        self.gf.SetSubVector(ardofs, elfun)
        coeff = NonlinearDerivativeCoefficient(self.gf)
        integ = mfem.MassIntegrator(coeff)
        integ.AssembleElementMatrix(el,T,elmat)

#### Asignamos valores en la frontera

In [ ]:
x = mfem.GridFunction(fespace)
x.Assign(1.)

#### Formulación variacional

In [ ]:
nform = mfem.NonlinearForm(fespace)
nonlin = NonlinearMassIntegrator(fespace)
nform.AddDomainIntegrator(nonlin)
nform.AddDomainIntegrator(mfem.DiffusionIntegrator())

b = mfem.LinearForm(fespace)
b.Assign(0.)

# Condiciones frontera sobre el segundo miembro
bdat = mfem.intArray([1]*mesh.bdr_attributes.Max())
nform.SetEssentialBC(bdat,b)

#### Resolución

In [ ]:
# 2. Configurar el Solver Lineal (interno para el paso de Newton)
lsolver = mfem.CGSolver()
lsolver.SetMaxIter(300)
lsolver.SetPrintLevel(0)

# 3. Configurar el NewtonSolver
newton_solver = mfem.NewtonSolver()
newton_solver.SetOperator(nform)
newton_solver.SetSolver(lsolver)
newton_solver.SetPrintLevel(1)
newton_solver.SetRelTol(1e-7)
newton_solver.Mult(b, x)    

#### Visualización

In [ ]:
glvis((mesh,x))